# Trajdata VR Stop Check

Load one `*_trajdata.npz` file, choose a random lap, and plot `VRtraj` as a function of lap-relative time. The red marker is placed at `endVR`, the trajectory field that marks when the animal reaches the end of the VR track. The gray dotted line shows `tstop`, the full lap/trial stop time, for comparison.

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.style.use('default')


In [ ]:
# Leave TRAJ_PATH as None to choose a random *_trajdata.npz file.
# Or set it to a specific file, for example:
# TRAJ_PATH = '../data/interim/VS100/2024-11-03/VS100_2024-11-03_14-40-11_trajdata.npz'
TRAJ_PATH = 'data/interim/VS103/2024-11-10/VS103_2024-11-10_14-58-13_trajdata.npz'

# Leave LAP_INDEX as None to choose a random lap. Otherwise use a 0-based lap index.
LAP_INDEX = None

# Set to an integer for reproducible random choices, or None for a new random choice each run.
RANDOM_SEED = None


In [ ]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    return cwd


def resolve_from_project(path_like, project_root: Path) -> Path:
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (project_root / path).resolve()


def decode_npz_json_scalar(arr: np.ndarray):
    return json.loads(arr.tobytes().decode('utf-8', errors='ignore'))


def load_lap_series(z, base_key: str):
    json_key = f'{base_key}__json'
    if json_key in z.files:
        return decode_npz_json_scalar(z[json_key])
    if base_key in z.files:
        return z[base_key]
    raise KeyError(f'Missing {base_key} / {json_key}')


PROJECT_ROOT = find_project_root()
TRAJ_ROOT = PROJECT_ROOT / 'data' / 'interim'
rng = random.Random(RANDOM_SEED)


In [ ]:
if TRAJ_PATH is None:
    traj_files = sorted(TRAJ_ROOT.rglob('*_trajdata.npz'))
    if not traj_files:
        raise FileNotFoundError(f'No *_trajdata.npz files found under {TRAJ_ROOT}')
    traj_path = rng.choice(traj_files)
else:
    traj_path = resolve_from_project(TRAJ_PATH, PROJECT_ROOT)

if not traj_path.exists():
    raise FileNotFoundError(traj_path)

with np.load(traj_path, allow_pickle=False) as z:
    required = ['traj__tstart', 'traj__tstop', 'traj__endVR']
    missing = [key for key in required if key not in z.files]
    if missing:
        raise KeyError(f'{traj_path.name} is missing required keys: {missing}')

    time_laps = load_lap_series(z, 'traj__time')
    vr_laps = load_lap_series(z, 'traj__VRtraj')
    tstart = np.asarray(z['traj__tstart'], dtype=float).ravel()
    tstop = np.asarray(z['traj__tstop'], dtype=float).ravel()
    end_vr = np.asarray(z['traj__endVR'], dtype=float).ravel()
    cond = np.asarray(z['traj__Cond']).ravel() if 'traj__Cond' in z.files else None
    wb = np.asarray(z['traj__WB']).astype(str).ravel() if 'traj__WB' in z.files else None

n_laps = len(vr_laps)
if len(time_laps) != n_laps:
    raise ValueError(f'time/VRtraj lap count mismatch: {len(time_laps)} vs {n_laps}')

if LAP_INDEX is None:
    lap_index = rng.randrange(n_laps)
else:
    lap_index = int(LAP_INDEX)
    if not 0 <= lap_index < n_laps:
        raise IndexError(f'LAP_INDEX={lap_index} outside 0..{n_laps - 1}')

time_rel = np.asarray(time_laps[lap_index], dtype=float).ravel()
vrtraj = np.asarray(vr_laps[lap_index], dtype=float).ravel()
if time_rel.size != vrtraj.size:
    n = min(time_rel.size, vrtraj.size)
    print(f'Warning: time/VRtraj length mismatch ({time_rel.size} vs {vrtraj.size}); truncating to {n}')
    time_rel = time_rel[:n]
    vrtraj = vrtraj[:n]

end_vr_rel = float(end_vr[lap_index] - tstart[lap_index])
tstop_rel = float(tstop[lap_index] - tstart[lap_index])

finite = np.isfinite(time_rel) & np.isfinite(vrtraj)
if not np.any(finite):
    raise ValueError(f'Lap {lap_index} has no finite time/VRtraj samples')

closest_idx = int(np.nanargmin(np.abs(time_rel[finite] - end_vr_rel)))
finite_indices = np.flatnonzero(finite)
marker_idx = int(finite_indices[closest_idx])
marker_time = float(time_rel[marker_idx])
marker_vr = float(vrtraj[marker_idx])

cond_label = f'Cond={cond[lap_index]}' if cond is not None else 'Cond=?'
wb_label = f'WB={wb[lap_index]}' if wb is not None else 'WB=?'

print(f'trajdata: {traj_path}')
print(f'lap: {lap_index} (0-based), {lap_index + 1} (1-based) / {n_laps}')
print(f'{cond_label}, {wb_label}')
print(f'tstart={tstart[lap_index]:.3f} s, endVR={end_vr[lap_index]:.3f} s, tstop={tstop[lap_index]:.3f} s')
print(f'endVR relative to lap start: {end_vr_rel:.3f} s')
print(f'closest plotted sample: t={marker_time:.3f} s, VRtraj={marker_vr:.3f}')
print(f'closest sample minus endVR: {marker_time - end_vr_rel:.6f} s')
print(f'VR at marker / lap max VR: {marker_vr:.3f} / {np.nanmax(vrtraj):.3f}')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
ax.plot(time_rel, vrtraj, color='tab:blue', lw=1.4, label='VRtraj')
ax.scatter(
    [marker_time],
    [marker_vr],
    s=90,
    color='crimson',
    edgecolor='white',
    linewidth=0.8,
    zorder=5,
    label='endVR / VR stop',
)
ax.axvline(marker_time, color='crimson', ls='--', lw=1.0, alpha=0.65)
ax.axvline(tstop_rel, color='0.45', ls=':', lw=1.2, alpha=0.8, label='tstop')

ax.set_title(f'{traj_path.name} | lap {lap_index + 1}/{n_laps} | {cond_label}, {wb_label}')
ax.set_xlabel('time from lap start (s)')
ax.set_ylabel('VRtraj')
ax.grid(True, alpha=0.25)
ax.legend(loc='best')
plt.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plotly_direction_colors = {
    'W': ('rgba(31, 119, 180, 0.35)', 'rgb(31, 119, 180)'),
    'B': ('rgba(255, 127, 14, 0.35)', 'rgb(255, 127, 14)'),
}
fallback_line_color = 'rgba(80, 80, 80, 0.35)'
fallback_marker_color = 'rgb(80, 80, 80)'


with np.load(traj_path, allow_pickle=False) as z:
    if 'traj__condition' in z.files:
        condition_labels = np.asarray(z['traj__condition']).astype(str).ravel()
    elif 'traj__condition__json' in z.files:
        condition_labels = np.asarray(decode_npz_json_scalar(z['traj__condition__json'])).astype(str).ravel()
    elif cond is not None:
        condition_labels = np.asarray([f'Cond={c}' for c in cond], dtype=str)
    else:
        condition_labels = np.asarray(['condition_unknown'] * n_laps, dtype=str)

if condition_labels.size < n_laps:
    padded = np.asarray(['condition_unknown'] * n_laps, dtype=str)
    padded[:condition_labels.size] = condition_labels
    condition_labels = padded
else:
    condition_labels = condition_labels[:n_laps]

condition_order = list(dict.fromkeys(condition_labels.tolist()))
n_conditions = len(condition_order)
condition_counts = {label: int(np.sum(condition_labels == label)) for label in condition_order}

fig = make_subplots(
    rows=n_conditions,
    cols=1,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=min(0.06, 0.22 / max(n_conditions, 1)),
    subplot_titles=[f'{label} | n_laps={condition_counts[label]}' for label in condition_order],
)

shown_direction_legend = set()
shown_marker_legend = False

for row, condition_label in enumerate(condition_order, start=1):
    lap_indices = np.flatnonzero(condition_labels == condition_label)
    condition_end_t = []
    condition_end_vr = []
    condition_end_hover = []
    condition_end_colors = []

    for i in lap_indices:
        t = np.asarray(time_laps[i], dtype=float).ravel()
        x = np.asarray(vr_laps[i], dtype=float).ravel()
        if t.size != x.size:
            n = min(t.size, x.size)
            t = t[:n]
            x = x[:n]

        finite = np.isfinite(t) & np.isfinite(x)
        if not np.any(finite):
            continue

        direction = wb[i] if wb is not None and i < len(wb) else '?'
        line_color, marker_color = plotly_direction_colors.get(direction, (fallback_line_color, fallback_marker_color))
        wb_text = f'WB={direction}'
        show_direction_legend = wb_text not in shown_direction_legend
        shown_direction_legend.add(wb_text)

        fig.add_trace(
            go.Scattergl(
                x=t[finite],
                y=x[finite],
                mode='lines',
                line=dict(color=line_color, width=1),
                name=wb_text,
                legendgroup=wb_text,
                showlegend=show_direction_legend,
                hovertemplate=(
                    f'condition={condition_label}<br>lap {i + 1}<br>{wb_text}<br>'
                    'time=%{x:.3f} s<br>VRtraj=%{y:.3f}<extra></extra>'
                ),
            ),
            row=row,
            col=1,
        )

        this_end_vr_rel = float(end_vr[i] - tstart[i])
        finite_indices = np.flatnonzero(finite)
        closest_idx = int(np.nanargmin(np.abs(t[finite] - this_end_vr_rel)))
        marker_idx = int(finite_indices[closest_idx])
        condition_end_t.append(float(t[marker_idx]))
        condition_end_vr.append(float(x[marker_idx]))
        condition_end_colors.append(marker_color)
        condition_end_hover.append(
            f'condition={condition_label}<br>lap {i + 1}<br>{wb_text}<br>'
            f'endVR={this_end_vr_rel:.3f} s<br>VRtraj={float(x[marker_idx]):.3f}'
        )

    fig.add_trace(
        go.Scatter(
            x=condition_end_t,
            y=condition_end_vr,
            mode='markers',
            marker=dict(size=7, color=condition_end_colors, line=dict(color='white', width=0.8)),
            name='endVR markers',
            legendgroup='endVR markers',
            showlegend=not shown_marker_legend,
            text=condition_end_hover,
            hovertemplate='%{text}<extra>endVR</extra>',
        ),
        row=row,
        col=1,
    )
    shown_marker_legend = True

    condition_end_rel = end_vr[lap_indices] - tstart[lap_indices]
    print(
        f'{condition_label}: n={len(lap_indices)}, '
        f'endVR median={np.nanmedian(condition_end_rel):.3f} s, '
        f'min={np.nanmin(condition_end_rel):.3f}, max={np.nanmax(condition_end_rel):.3f}'
    )

fig.update_layout(
    title=f'Interactive lap overlays by condition | {traj_path.name}',
    template='plotly_white',
    height=max(360, 260 * n_conditions),
    hovermode='closest',
    legend=dict(groupclick='togglegroup'),
)
fig.update_xaxes(title_text='time from lap start (s)', row=n_conditions, col=1, rangeslider_visible=True)
for row in range(1, n_conditions + 1):
    fig.update_yaxes(title_text='VRtraj', row=row, col=1)

fig.show(config={'scrollZoom': True, 'displaylogo': False})


In [ ]:
# Interactive condition-split overlay, keeping only samples below this speed threshold.
SPEED_THRESHOLD = 1.0
USE_ABSOLUTE_SPEED = True

import plotly.graph_objects as go
from plotly.subplots import make_subplots

plotly_direction_colors = {
    'W': ('rgba(31, 119, 180, 0.35)', 'rgb(31, 119, 180)'),
    'B': ('rgba(255, 127, 14, 0.35)', 'rgb(255, 127, 14)'),
}
fallback_line_color = 'rgba(80, 80, 80, 0.35)'
fallback_marker_color = 'rgb(80, 80, 80)'


with np.load(traj_path, allow_pickle=False) as z:
    if 'traj__Speed__json' in z.files or 'traj__Speed' in z.files:
        speed_laps = load_lap_series(z, 'traj__Speed')
        speed_field_used = 'traj__Speed'
    elif 'traj__XSpeed__json' in z.files or 'traj__XSpeed' in z.files:
        speed_laps = load_lap_series(z, 'traj__XSpeed')
        speed_field_used = 'traj__XSpeed'
    else:
        raise KeyError(f'{traj_path.name} is missing traj__Speed / traj__XSpeed')

    if 'traj__condition' in z.files:
        speed_condition_labels = np.asarray(z['traj__condition']).astype(str).ravel()
    elif 'traj__condition__json' in z.files:
        speed_condition_labels = np.asarray(decode_npz_json_scalar(z['traj__condition__json'])).astype(str).ravel()
    elif cond is not None:
        speed_condition_labels = np.asarray([f'Cond={c}' for c in cond], dtype=str)
    else:
        speed_condition_labels = np.asarray(['condition_unknown'] * n_laps, dtype=str)

if speed_condition_labels.size < n_laps:
    padded = np.asarray(['condition_unknown'] * n_laps, dtype=str)
    padded[:speed_condition_labels.size] = speed_condition_labels
    speed_condition_labels = padded
else:
    speed_condition_labels = speed_condition_labels[:n_laps]

speed_condition_order = list(dict.fromkeys(speed_condition_labels.tolist()))
n_speed_conditions = len(speed_condition_order)
speed_condition_counts = {label: int(np.sum(speed_condition_labels == label)) for label in speed_condition_order}

fig = make_subplots(
    rows=n_speed_conditions,
    cols=1,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=min(0.06, 0.22 / max(n_speed_conditions, 1)),
    subplot_titles=[f'{label} | n_laps={speed_condition_counts[label]}' for label in speed_condition_order],
)

shown_direction_legend = set()
shown_marker_legend = False
speed_msg = f'Using {speed_field_used}; plotting samples with speed < {SPEED_THRESHOLD:g}'
if USE_ABSOLUTE_SPEED:
    speed_msg = f'Using {speed_field_used}; plotting samples with abs(speed) < {SPEED_THRESHOLD:g}'
print(speed_msg)

for row, condition_label in enumerate(speed_condition_order, start=1):
    lap_indices = np.flatnonzero(speed_condition_labels == condition_label)
    condition_end_t = []
    condition_end_vr = []
    condition_end_hover = []
    condition_end_colors = []
    slow_samples = 0
    total_samples = 0
    laps_with_slow_samples = 0

    for i in lap_indices:
        t = np.asarray(time_laps[i], dtype=float).ravel()
        x = np.asarray(vr_laps[i], dtype=float).ravel()
        speed = np.asarray(speed_laps[i], dtype=float).ravel()
        n = min(t.size, x.size, speed.size)
        t = t[:n]
        x = x[:n]
        speed = speed[:n]

        speed_for_threshold = np.abs(speed) if USE_ABSOLUTE_SPEED else speed
        finite = np.isfinite(t) & np.isfinite(x) & np.isfinite(speed_for_threshold)
        slow = finite & (speed_for_threshold < SPEED_THRESHOLD)
        total_samples += int(np.sum(finite))
        slow_samples += int(np.sum(slow))
        if not np.any(slow):
            continue
        laps_with_slow_samples += 1

        x_masked = x.astype(float).copy()
        x_masked[~slow] = np.nan

        direction = wb[i] if wb is not None and i < len(wb) else '?'
        line_color, marker_color = plotly_direction_colors.get(direction, (fallback_line_color, fallback_marker_color))
        wb_text = f'WB={direction}'
        show_direction_legend = wb_text not in shown_direction_legend
        shown_direction_legend.add(wb_text)

        fig.add_trace(
            go.Scattergl(
                x=t,
                y=x_masked,
                customdata=speed_for_threshold.reshape(-1, 1),
                mode='lines',
                line=dict(color=line_color, width=1.4),
                name=wb_text,
                legendgroup=wb_text,
                showlegend=show_direction_legend,
                hovertemplate=(
                    f'condition={condition_label}<br>lap {i + 1}<br>{wb_text}<br>'
                    'time=%{x:.3f} s<br>VRtraj=%{y:.3f}<br>speed=%{customdata[0]:.3f}<extra></extra>'
                ),
            ),
            row=row,
            col=1,
        )

        this_end_vr_rel = float(end_vr[i] - tstart[i])
        finite_indices = np.flatnonzero(finite)
        closest_idx = int(np.nanargmin(np.abs(t[finite] - this_end_vr_rel)))
        marker_idx = int(finite_indices[closest_idx])
        if slow[marker_idx]:
            condition_end_t.append(float(t[marker_idx]))
            condition_end_vr.append(float(x[marker_idx]))
            condition_end_colors.append(marker_color)
            condition_end_hover.append(
                f'condition={condition_label}<br>lap {i + 1}<br>{wb_text}<br>'
                f'endVR={this_end_vr_rel:.3f} s<br>VRtraj={float(x[marker_idx]):.3f}<br>speed={float(speed_for_threshold[marker_idx]):.3f}'
            )

    if condition_end_t:
        fig.add_trace(
            go.Scatter(
                x=condition_end_t,
                y=condition_end_vr,
                mode='markers',
                marker=dict(size=7, color=condition_end_colors, line=dict(color='white', width=0.8)),
                name='endVR markers below threshold',
                legendgroup='endVR markers below threshold',
                showlegend=not shown_marker_legend,
                text=condition_end_hover,
                hovertemplate='%{text}<extra>endVR</extra>',
            ),
            row=row,
            col=1,
        )
        shown_marker_legend = True

    slow_fraction = slow_samples / total_samples if total_samples else np.nan
    print(
        f'{condition_label}: laps with any speed<{SPEED_THRESHOLD:g} = {laps_with_slow_samples}/{len(lap_indices)}, '
        f'slow sample fraction={slow_fraction:.3%}'
    )

fig.update_layout(
    title=f'Condition overlays with speed < {SPEED_THRESHOLD:g} | {traj_path.name}',
    template='plotly_white',
    height=max(360, 260 * n_speed_conditions),
    hovermode='closest',
    legend=dict(groupclick='togglegroup'),
)
fig.update_xaxes(title_text='time from lap start (s)', row=n_speed_conditions, col=1, rangeslider_visible=True)
for row in range(1, n_speed_conditions + 1):
    fig.update_yaxes(title_text='VRtraj', row=row, col=1)

fig.show(config={'scrollZoom': True, 'displaylogo': False})


In [ ]:
# Rank trials by the fraction of pre-endVR samples with speed below threshold.
# This excludes the reward period/stillness after the animal reached the VR end.
RANK_SPEED_THRESHOLD = SPEED_THRESHOLD if 'SPEED_THRESHOLD' in globals() else 1.0
RANK_USE_ABSOLUTE_SPEED = True

import pandas as pd
import plotly.express as px
from IPython.display import display

with np.load(traj_path, allow_pickle=False) as z:
    if 'traj__Speed__json' in z.files or 'traj__Speed' in z.files:
        rank_speed_laps = load_lap_series(z, 'traj__Speed')
        rank_speed_field_used = 'traj__Speed'
    elif 'traj__XSpeed__json' in z.files or 'traj__XSpeed' in z.files:
        rank_speed_laps = load_lap_series(z, 'traj__XSpeed')
        rank_speed_field_used = 'traj__XSpeed'
    else:
        raise KeyError(f'{traj_path.name} is missing traj__Speed / traj__XSpeed')

    if 'traj__condition' in z.files:
        rank_condition_labels = np.asarray(z['traj__condition']).astype(str).ravel()
    elif 'traj__condition__json' in z.files:
        rank_condition_labels = np.asarray(decode_npz_json_scalar(z['traj__condition__json'])).astype(str).ravel()
    elif cond is not None:
        rank_condition_labels = np.asarray([f'Cond={c}' for c in cond], dtype=str)
    else:
        rank_condition_labels = np.asarray(['condition_unknown'] * n_laps, dtype=str)

if rank_condition_labels.size < n_laps:
    padded = np.asarray(['condition_unknown'] * n_laps, dtype=str)
    padded[:rank_condition_labels.size] = rank_condition_labels
    rank_condition_labels = padded
else:
    rank_condition_labels = rank_condition_labels[:n_laps]

rank_rows = []
for i in range(n_laps):
    t = np.asarray(time_laps[i], dtype=float).ravel()
    speed = np.asarray(rank_speed_laps[i], dtype=float).ravel()
    n = min(t.size, speed.size)
    t = t[:n]
    speed = speed[:n]

    end_vr_rel_i = float(end_vr[i] - tstart[i])
    speed_for_threshold = np.abs(speed) if RANK_USE_ABSOLUTE_SPEED else speed
    pre_endvr = t <= end_vr_rel_i
    valid = pre_endvr & np.isfinite(t) & np.isfinite(speed_for_threshold)
    slow = valid & (speed_for_threshold < RANK_SPEED_THRESHOLD)

    valid_n = int(np.sum(valid))
    slow_n = int(np.sum(slow))
    slow_fraction = slow_n / valid_n if valid_n else np.nan
    pre_endvr_duration_s = float(np.nanmax(t[valid]) - np.nanmin(t[valid])) if valid_n else np.nan

    rank_rows.append(
        {
            'rank': np.nan,
            'lap_index_0b': i,
            'lap_index_1b': i + 1,
            'condition': rank_condition_labels[i],
            'WB': wb[i] if wb is not None and i < len(wb) else '?',
            'pre_endVR_s': end_vr_rel_i,
            'pre_endVR_duration_s': pre_endvr_duration_s,
            'valid_pre_endVR_samples': valid_n,
            'slow_pre_endVR_samples': slow_n,
            'slow_pre_endVR_fraction': slow_fraction,
            'slow_pre_endVR_percent': 100.0 * slow_fraction if np.isfinite(slow_fraction) else np.nan,
            'speed_threshold': RANK_SPEED_THRESHOLD,
            'speed_field': rank_speed_field_used,
        }
    )

rank_df = pd.DataFrame(rank_rows).sort_values(
    ['slow_pre_endVR_fraction', 'pre_endVR_s'],
    ascending=[False, False],
    na_position='last',
).reset_index(drop=True)
rank_df['rank'] = np.arange(1, len(rank_df) + 1)

print(
    f'Ranking uses {rank_speed_field_used}; '
    f'pre-endVR samples only; '
    f"threshold={'abs(speed)' if RANK_USE_ABSOLUTE_SPEED else 'speed'} < {RANK_SPEED_THRESHOLD:g}"
)
display(
    rank_df[
        [
            'rank',
            'lap_index_1b',
            'condition',
            'WB',
            'slow_pre_endVR_percent',
            'pre_endVR_s',
            'valid_pre_endVR_samples',
            'slow_pre_endVR_samples',
        ]
    ].style.format(
        {
            'slow_pre_endVR_percent': '{:.2f}',
            'pre_endVR_s': '{:.3f}',
        }
    )
)

fig = px.bar(
    rank_df,
    x='rank',
    y='slow_pre_endVR_percent',
    color='condition',
    pattern_shape='WB',
    hover_data={
        'lap_index_1b': True,
        'condition': True,
        'WB': True,
        'slow_pre_endVR_percent': ':.2f',
        'pre_endVR_s': ':.3f',
        'valid_pre_endVR_samples': True,
        'slow_pre_endVR_samples': True,
        'rank': True,
    },
    title=f'Trials ranked by pre-endVR slow fraction | speed < {RANK_SPEED_THRESHOLD:g} | {traj_path.name}',
    labels={
        'rank': 'trial rank',
        'slow_pre_endVR_percent': '% pre-endVR samples below speed threshold',
    },
)
fig.update_traces(marker_line_width=0)
fig.update_layout(template='plotly_white', height=520)
fig.show(config={'scrollZoom': True, 'displaylogo': False})
